# RG3 미성형 D 상태별 과거 비라벨 기준선


In [ ]:
from pathlib import Path
import os,json,nbformat,joblib
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd();WORK=ROOT/'rg3/internal';tables=WORK/'reports/tables'
package=nbformat.read(ROOT/'rg3/01_preprocess_train.ipynb',4);exec(package.cells[1].source)
prev=Path.cwd()
try:
 os.chdir(WORK)
 setup=source('145_rg3_shortshot_unlabeled_application.ipynb');base={'__name__':'__main__'};exec(setup[:setup.index('u_all=pd.read_csv')],base)
finally:os.chdir(prev)
lab=base['lab'];y=base['yn'];g=base['gn'];use=base['usable'];normal=use&(y==0)
dense=base['nc'][0]['dense'];hopper=-dense.Hopper_Temperature__dev10.to_numpy()
physical=base['nb']['views']['physical12'];joint=base['nd']['jointbase']
mechanical_indices=[0,1,2,3,4,5,6,7,8,10,11]
rawtemp=[f'Barrel_Temperature_{i}' for i in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
context=lab[[v+'_raw' for v in rawtemp]].to_numpy(float)
oldpred=pd.read_csv(tables/'rg3_204_shortshot_reduction_predictions.csv',parse_dates=['TimeStamp'])
oldpred=oldpred[oldpred.candidate.eq('pruned15')].set_index('TimeStamp').loc[lab.TimeStamp]
unknown=pd.read_csv(tables/'rg3_207_dense_unknown_history.csv',parse_dates=['TimeStamp'])
diag=pd.read_csv(tables/'rg3_207_shortshot_alarm_diagnostics.csv',parse_dates=['TimeStamp'])
u=unknown.set_index('TimeStamp').loc[diag.TimeStamp].reset_index()
ratio=lambda a,b:a/b.where(b.abs()>1e-6)
phyU=np.column_stack([u.Max_Injection_Pressure-u.Max_Switch_Over_Pressure,ratio(u.Max_Switch_Over_Pressure,u.Max_Injection_Pressure),ratio(u.Filling_Time,u.Injection_Time),ratio(u.Filling_Time,u.Max_Injection_Speed),u.Plasticizing_Position-u.Cushion_Position,u.Plasticizing_Position__delta1-u.Cushion_Position__delta1,ratio(u.Plasticizing_Time,u.Cycle_Time),u.Plasticizing_Time__sparse_dev,u.Average_Back_Pressure__sparse_dev,u.Barrel_Temperature_5-(u.Mold_Temperature_3+u.Mold_Temperature_4)/2,u.Max_Injection_Pressure__sparse_dev,u.Cushion_Position__sparse_dev])
jointU=np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Hopper_Temperature__past10_std,u.Max_Injection_Speed__past10_std,u.Cushion_Position__past10_std,u.Plasticizing_Time__past10_std,u.Barrel_Temperature_5__past10_std,u.Average_Back_Pressure__past10_std])
ctxU=u[rawtemp].to_numpy(float)

# Past unlabeled rows are used ONLY for robust baseline statistics, never assigned a normal class target.
cutoff=lab.TimeStamp.min()-pd.Timedelta(hours=1)
full_u=pd.read_csv(tables/'rg3_207_dense_unknown_history.csv',parse_dates=['TimeStamp'])
all_joint=np.column_stack([full_u.Hopper_Temperature__dev10,full_u.Barrel_Temperature_4__dev10,full_u.Barrel_Temperature_5__delta1,full_u.Barrel_Temperature_3__delta1,full_u.Mold_Temperature_4__delta1,full_u.Hopper_Temperature__past10_std,full_u.Max_Injection_Speed__past10_std,full_u.Cushion_Position__past10_std,full_u.Plasticizing_Time__past10_std,full_u.Barrel_Temperature_5__past10_std,full_u.Average_Back_Pressure__past10_std])
poolmask=full_u.TimeStamp.lt(cutoff).to_numpy() & full_u.history_count.ge(10).to_numpy() & ~full_u.sensor_conflict.to_numpy() & np.isfinite(all_joint).all(axis=1) & np.isfinite(full_u[rawtemp].to_numpy(float)).all(axis=1)
pool_joint=all_joint[poolmask];pool_context=full_u.loc[poolmask,rawtemp].to_numpy(float);pool_times=full_u.TimeStamp.to_numpy()[poolmask]
profile_names=['Injection_Time','Cycle_Time','Cushion_Position','Plasticizing_Position']
pool_profile=full_u.loc[poolmask,profile_names].to_numpy(float)
def pool_selection(ref):
 selected=np.ones(len(pool_times),bool)
 for j,v in enumerate(profile_names):
  values=lab.iloc[ref].loc[y[ref]==0,v+'_raw'].to_numpy(float)
  med=np.median(values);iqr=np.quantile(values,.75)-np.quantile(values,.25);tol=max(abs(med)*.1,10*iqr,.01)
  selected &= (pool_profile[:,j]>=med-tol)&(pool_profile[:,j]<=med+tol)
 return selected
assert len(pool_times)>0 and pd.Timestamp(pool_times.max())<lab.TimeStamp.min()
memo={}
def conditioned(ref,qjoint,qcontext,qtimes):
 normal_ids=ref[y[ref]==0]
 selected=pool_selection(ref) # Selection uses THIS training reference's normal labels only, never heldout labels.
 refs=np.vstack([joint[normal_ids],pool_joint[selected]]);ctx=np.vstack([context[normal_ids],pool_context[selected]])
 times=np.r_[lab.TimeStamp.to_numpy()[normal_ids],pool_times[selected]]
 ci=SimpleImputer(strategy='median').fit(ctx);ctx=ci.transform(ctx)
 sc=RobustScaler().fit(ctx);ctx=sc.transform(ctx)
 ri=SimpleImputer(strategy='median').fit(refs);refs=ri.transform(refs);query=ri.transform(qjoint)
 nn=NearestNeighbors(n_neighbors=min(101,len(refs))).fit(ctx)
 near=nn.kneighbors(sc.transform(ci.transform(qcontext)),return_distance=False)
 floor=np.maximum(np.std(refs,axis=0)*.05,np.array([.1,.1,.1,.1,.1,.01,.01,.01,.01,.01,.01]))
 result=[]
 for row,ids,ts in zip(query,near,qtimes):
  ids=ids[times[ids]!=np.datetime64(ts)][:100]
  center=np.median(refs[ids],axis=0)
  spread=np.maximum(1.4826*np.median(np.abs(refs[ids]-center),axis=0),floor)
  result.append(np.clip((row-center)/spread,-12,12))
 return np.asarray(result)
def features(x):
 j=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))
 return np.column_stack([j.max(axis=1),np.sort(j,axis=1)[:,-2],j.mean(axis=1),(j>1).sum(axis=1)])
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(g[ref]).isdisjoint(g[q])
 a=conditioned(ref,joint[ref],context[ref],lab.TimeStamp.to_numpy()[ref])
 b=conditioned(ref,joint[q],context[q],lab.TimeStamp.to_numpy()[q])
 model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(features(a),y[ref])
 memo[key]=model.predict_proba(features(b))[:,1]
 return memo[key]
records=[];audit=[]
for block in pd.unique(g):
 q=np.flatnonzero(g==block);ref=np.flatnonzero((g!=block)&use)
 outer=predict(ref,q);inner=np.full(len(ref),np.nan)
 for ib in pd.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 abc=oldpred.iloc[q][['A','B','C']].any(axis=1).to_numpy()
 hp=hopper[ref];hcut=float(np.quantile(hp[(y[ref]==0)&np.isfinite(hp)],.95,method='higher'))
 for quant in [.99,.995,.999]:
  dcut=float(np.quantile(inner[y[ref]==0],quant,method='higher'))
  dd=(outer>=dcut)&(hopper[q]>=hcut)
  for j,idx in enumerate(q):records.append(dict(policy=str(quant),TimeStamp=lab.TimeStamp.iloc[idx],type=lab.type.iloc[idx],actual=int(y[idx]),prediction=int(abc[j]|dd[j]),D=int(dd[j]),D_score=float(outer[j]),cutoff=dcut))
 audit.append(dict(block=block,overlap=len(set(lab.TimeStamp.iloc[q])&set(lab.TimeStamp.iloc[ref])),past_unlabeled_reference=len(pool_times),reference_latest=str(pd.Timestamp(pool_times.max()))))
 print('State-conditioned D completed',block,flush=True)
pp=pd.DataFrame(records);metrics=[]
for policy,a in pp[pp.type.isin(['정상','미성형'])].groupby('policy'):
 tn,fp,fn,tp=confusion_matrix(a.actual,a.prediction,labels=[0,1]).ravel()
 metrics.append(dict(policy=policy,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn))))
ref=np.flatnonzero(use);inner=np.full(len(ref),np.nan)
for ib in pd.unique(g[ref]):
 mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
a=conditioned(ref,joint[ref],context[ref],lab.TimeStamp.to_numpy()[ref])
b=conditioned(ref,jointU,ctxU,u.TimeStamp.to_numpy())
model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(features(a),y[ref])
scores=model.predict_proba(features(b))[:,1]
abc=diag[['A','B','C']].any(axis=1).to_numpy()
hp=hopper[ref];hcut=float(np.quantile(hp[(y[ref]==0)&np.isfinite(hp)],.95,method='higher'))
um=[]
for quant in [.99,.995,.999]:
 dcut=float(np.quantile(inner[y[ref]==0],quant,method='higher'))
 dd=(scores>=dcut)&(-u.Hopper_Temperature__dev10.to_numpy()>=hcut);pr=abc|dd
 um.append(dict(policy=str(quant),N=len(u),candidates=int(pr.sum()),D=int(dd.sum()),alarm_percent=float(100*pr.mean())))
 pd.DataFrame({'TimeStamp':u.TimeStamp,'prediction':pr.astype(int),'D':dd.astype(int),'D_score':scores}).to_csv(tables/('rg3_209_conditioned_unlabeled_'+str(quant)+'.csv'),index=False)
mm=pd.DataFrame(metrics);um=pd.DataFrame(um)
pp.to_csv(tables/'rg3_209_conditioned_D_predictions.csv',index=False)
mm.to_csv(tables/'rg3_209_conditioned_D_metrics.csv',index=False)
um.to_csv(tables/'rg3_209_conditioned_D_unlabeled_metrics.csv',index=False)
pd.DataFrame(audit).to_csv(tables/'rg3_209_conditioned_D_audit.csv',index=False)
assert all(a['overlap']==0 for a in audit)
print(mm.to_string(index=False));print(um.to_string(index=False))
text='# RG3 상태별 D 기준선 재학습\n\n과거비라벨을품질라벨로붙이지않고기준선의강건중앙값/MAD계산에만사용한다. 비라벨참조는최초공식라벨샷1시간이전으로고정. 온도9개의유사한상태100개이웃에서동시변화11개피처를정규화한다. 분류기는정상/미성형실제라벨만학습한다. ABC15모델의기존세가지출력은고정했다.\n\n'+mm.to_string(index=False)+'\n\n'+um.to_string(index=False)
text+='\n\n비라벨6590샷은실제정답없음. 과거비라벨모드에서대다수가정상동작이라는가정을쓰므로기준선이불량을흡수할위험이있다. 역사적비라벨에학습후모델을소급적용한참고경보율이며새독립성능/실시간미래검증이아니다. 쿼리자신의timestamp는이웃기준선에서제외했다. 과거U내예측은동시대다른U참조를쓸수있어엄격한온라인시점검증이아니다. 배포모델및Git미교체.\n'
(WORK/'reports/rg3_209_conditioned_D_review.md').write_text(text,encoding='utf-8')

